# Structured Outputs
This notebook demonstrates how to work with different output formats and parsers when working with Language Models (LLM).

## What we'll learn:
- Basic string output parsing
- Working with tools and structured outputs
- Using Pydantic models for type validation
- Understanding different parser types and their use cases

### Setup

In [1]:

import os
import json
from datetime import datetime
from pydantic import BaseModel, Field
from lib.agents import Agent
from lib.state_machine import Run
from typing import List,Any, Annotated
from lib.llm import LLM
from tavily import TavilyClient
from typing import Dict
from lib.tooling import tool
from lib.messages import UserMessage, SystemMessage,BaseMessage
from lib.evaluation import TestCase, AgentEvaluator, EvaluationResult
from dotenv import load_dotenv
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG
from lib.parsers import  JsonOutputParser,PydanticOutputParser

## Setup

In [2]:
load_dotenv()

True

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## Load data into Vector DB

In [ ]:
db = VectorStoreManager(OPENAI_API_KEY)
print("VectorStoreManager initialized")


VectorStoreManager():<chromadb.api.client.Client object at 0x00000219D8283B60>

In [ ]:
loader_service = CorpusLoaderService(db)

collection_name = "games_market"
existing_store = db.get_store(collection_name)
if existing_store is not None:
    print(f"Using existing collection: {collection_name}")
    games_store = existing_store
else:
    print(f"Collection '{collection_name}' not found; loading from PDF...")
    games_store = loader_service.load_pdf(
        store_name=collection_name,
        pdf_path="TheGamingIndustry2024.pdf",
    )

In [6]:
rag_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,
)

In [ ]:
games_market_rag = RAG(
    llm=rag_llm,
    vector_store=games_store
)

VectorStore `games_market` ready!
Pages from `TheGamingIndustry2024.pdf` added!


In [8]:
result:Run = games_market_rag.invoke("What's the  state of virtual reality")
print(result.get_final_state()["answer"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
The state of virtual reality (VR) in 2024 is characterized by its significant role in the gaming industry, where it allows gamers to immerse themselves in expansive digital worlds. VR technology is enhancing gaming experiences by enabling realistic and interactive environments, as exemplified by games like "Half-Life: Alyx." This technology incorporates high-fidelity graphics, intuitive controls, and immersive audio, making gameplay more engaging and lifelike. Additionally, VR is being leveraged beyond entertainment, finding applications in education, therapy, and social change, indicating its growing influence and versatility in various fields.


## Tools

In [9]:
@tool
def retrieve_game(query):
    """
    Search the vector database for relevant information.
    
    Source: The Gaming Industry in 2024 - Trends, Technologies & Predictions
    Publisher: Ixie Gaming
    Release: 2024
    ```
        The gaming industry, on the brink of transformative change due to technological
        advancements, is redefining entertainment and social interaction with
        immersive, personalized, and interactive experiences.
    ```

    args:
        query (str): Search query
    """
    result:Run = games_market_rag.invoke(query)
    return result.get_final_state()["answer"]

In [ ]:
@tool
def evaluate_retrieval(user_query: str, agent_response: str, retrieved_docs: str = "") -> Dict:
    """
    Evaluate the agent response against the user query using AgentEvaluator.

    args:
        user_query: the user query text
        agent_response: the response text from the agent
        retrieved_docs: optional related documents for context
    """
    print(f"\n[evaluate_retrieval] user_query={user_query}")
    print(f"[evaluate_retrieval] agent_response={agent_response}")
    if retrieved_docs:
        print(f"[evaluate_retrieval] retrieved_docs={retrieved_docs}")

    test_case = TestCase(
        id="eval_retrieval_1",
        description=f"Evaluate retrieval response for query: {user_query}",
        user_query=user_query,
        expected_tools=[],
        reference_answer=None,
        max_steps=3
    )

    evaluator = AgentEvaluator()
    try:
        evaluation_result = evaluator.evaluate_final_response(
            test_case=test_case,
            agent_response=agent_response,
            execution_time=0.0,
            total_tokens=0
        )
        return evaluation_result.model_dump() if hasattr(evaluation_result, 'model_dump') else evaluation_result.dict()
    except Exception as e:
        print(f"[evaluate_retrieval] AgentEvaluator failed: {e}")

    coverage_score = 1.0 if user_query.lower().split()[0] in agent_response.lower() else 0.5
    precision_score = 1.0 if len(agent_response) > 10 else 0.3

    return {
        "user_query": user_query,
        "agent_response": agent_response,
        "retrieved_docs": retrieved_docs,
        "coverage_score": coverage_score,
        "precision_score": precision_score,
        "overall_score": (coverage_score + precision_score) / 2,
        "comment": "Fallback evaluation; AgentEvaluator unavailable."
    }

In [11]:

@tool
def game_web_search(query: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API
    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    """
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)
    
    # Perform the search
    search_result = client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )
    
    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }
    
    return formatted_results

In [12]:
agentic_rag = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game,evaluate_retrieval,game_web_search],
    instructions=(
        "You are an Agentic RAG assistant that can intelligently decide which tools to use "        
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not."
        "Perform a web search if the retrieved documents are not enough to answer the question, and use the retrieved information to decide the best query to perform the web search."
        "provide confidence levels for your answers and explain your reasoning in detail."
        "provide sources for your answers and explain your reasoning in detail."
        
    )    
)

In [13]:
def print_messages(messages: List[BaseMessage]):
    for m in messages: 
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

In [14]:
run_1 = agentic_rag.invoke(
    query= (
        "Who developed FIFA 21?"
    ),
    session_id="games1",
)

print("\nMessages from run 1:")
messages = run_1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are an Agentic RAG assistant that can intelligently decide which tools to use Your task is to evaluate if the documents are enough to respond the query. Give a detailed explanation, so it's possible to take an action to accept it or not.Perform a web search if the retrieved documents are not enough to answer the question, and use the retrieved information to decide the best query to perform the 

In [15]:
run_2 = agentic_rag.invoke(
    query= (
        "When was God of War Ragnarok released?"
    ),
    session_id="games2",
)

print("\nMessages from run 2:")
messages = run_2.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 2:
 -> (role = system, content = You are an Agentic RAG assistant that can intelligently decide which tools to use Your task is to evaluate if the documents are enough to respond the query. Give a detailed explanation, so it's possible to take an action to accept it or not.Perform a web search if the retrieved documents are not enough to answer the question, and use the retrieved information to decide the best query to perform the web search.provide confidence levels for your answers and explain your reasoning in detail.provide sources for your answers and explain your reasoning in detail., tool_calls = None)
 -> (role = user, content = When was God of War Ragnarok released?, tool_calls = None)
 -> (role = assista

In [16]:
run_3 = agentic_rag.invoke(
    query= (
        "What platform was Pokémon Red launched on?"
    ),
    session_id="games3",
)

print("\nMessages from run 3:")
messages = run_3.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 3:
 -> (role = system, content = You are an Agentic RAG assistant that can intelligently decide which tools to use Your task is to evaluate if the documents are enough to respond the query. Give a detailed explanation, so it's possible to take an action to accept it or not.Perform a web search if the retrieved documents are not enough to answer the question, and use the retrieved information to decide the best query to perform the 